In [1]:
# 기본 라이브러리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False
import gc


# 인코더 추가
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
#VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
#상수항추가
from statsmodels.tools.tools import add_constant
# 카이제곱, ANOVA
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway
#Turkeyhsd
from statsmodels.stats.multicomp import pairwise_tukeyhsd

### 데이터 불러오기

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
# parquet 파일 데이터를 읽어온다.
# x 값으로 쓸 데이터
df1_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201807_train_승인매출정보.parquet')
df2_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201808_train_승인매출정보.parquet')
df3_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201809_train_승인매출정보.parquet')
df4_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201810_train_승인매출정보.parquet')
df5_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201811_train_승인매출정보.parquet')
df6_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201812_train_승인매출정보.parquet')

In [4]:
# 결과데이터를 불러온다.
segment_df = pd.read_csv('/content/drive/MyDrive/Segment.csv')
segment_df

,기준년월,ID,Segment
0,201807,TRAIN_000000,D
1,201807,TRAIN_000001,E
2,201807,TRAIN_000002,C
3,201807,TRAIN_000003,D
4,201807,TRAIN_000004,E
...,...,...,...
2399995,201812,TRAIN_399995,E
2399996,201812,TRAIN_399996,D
2399997,201812,TRAIN_399997,C
2399998,201812,TRAIN_399998,E


In [20]:
# 데이터프레임의 컬럼 이름을 리스트에 담는다.
column_list = df1_train.columns.tolist()

In [21]:
print(column_list)

['기준년월', 'ID', '최종이용일자_기본', '최종이용일자_신판', '최종이용일자_CA', '최종이용일자_카드론', '최종이용일자_체크', '최종이용일자_일시불', '최종이용일자_할부', '이용건수_신용_B0M', '이용건수_신판_B0M', '이용건수_일시불_B0M', '이용건수_할부_B0M', '이용건수_할부_유이자_B0M', '이용건수_할부_무이자_B0M', '이용건수_부분무이자_B0M', '이용건수_CA_B0M', '이용건수_체크_B0M', '이용건수_카드론_B0M', '이용금액_일시불_B0M', '이용금액_할부_B0M', '이용금액_할부_유이자_B0M', '이용금액_할부_무이자_B0M', '이용금액_부분무이자_B0M', '이용금액_CA_B0M', '이용금액_체크_B0M', '이용금액_카드론_B0M', '이용후경과월_신용', '이용후경과월_신판', '이용후경과월_일시불', '이용후경과월_할부', '이용후경과월_할부_유이자', '이용후경과월_할부_무이자', '이용후경과월_부분무이자', '이용후경과월_CA', '이용후경과월_체크', '이용후경과월_카드론', '이용건수_신용_R12M', '이용건수_신판_R12M', '이용건수_일시불_R12M', '이용건수_할부_R12M', '이용건수_할부_유이자_R12M', '이용건수_할부_무이자_R12M', '이용건수_부분무이자_R12M', '이용건수_CA_R12M', '이용건수_체크_R12M', '이용건수_카드론_R12M', '이용금액_일시불_R12M', '이용금액_할부_R12M', '이용금액_할부_유이자_R12M', '이용금액_할부_무이자_R12M', '이용금액_부분무이자_R12M', '이용금액_CA_R12M', '이용금액_체크_R12M', '이용금액_카드론_R12M', '최대이용금액_일시불_R12M', '최대이용금액_할부_R12M', '최대이용금액_할부_유이자_R12M', '최대이용금액_할부_무이자_R12M', '최대이용금액_부분무이자_R12M', '최대이용금액_CA_R12M', '최대이용금액_체크_R12M', '

In [22]:
# 슬라이싱
column_list = ['기준년월', 'ID', '이용가맹점수', '이용금액_해외', '쇼핑_도소매_이용금액', '쇼핑_백화점_이용금액', '쇼핑_마트_이용금액', '쇼핑_슈퍼마켓_이용금액', '쇼핑_편의점_이용금액', '쇼핑_아울렛_이용금액', '쇼핑_온라인_이용금액', '쇼핑_기타_이용금액', '교통_주유이용금액', '교통_정비이용금액', '교통_통행료이용금액', '교통_버스지하철이용금액', '교통_택시이용금액', '교통_철도버스이용금액', '여유_운동이용금액', '여유_Pet이용금액', '여유_공연이용금액', '여유_공원이용금액', '여유_숙박이용금액', '여유_여행이용금액', '여유_항공이용금액', '여유_기타이용금액', '납부_통신비이용금액', '납부_관리비이용금액', '납부_렌탈료이용금액', '납부_가스전기료이용금액', '납부_보험료이용금액', '납부_유선방송이용금액', '납부_건강연금이용금액', '납부_기타이용금액', '_1순위업종', '_1순위업종_이용금액', '_2순위업종', '_2순위업종_이용금액', '_3순위업종', '_3순위업종_이용금액', '_1순위쇼핑업종', '_1순위쇼핑업종_이용금액', '_2순위쇼핑업종', '_2순위쇼핑업종_이용금액', '_3순위쇼핑업종', '_3순위쇼핑업종_이용금액', '_1순위교통업종', '_1순위교통업종_이용금액', '_2순위교통업종', '_2순위교통업종_이용금액', '_3순위교통업종', '_3순위교통업종_이용금액', '_1순위여유업종', '_1순위여유업종_이용금액', '_2순위여유업종', '_2순위여유업종_이용금액', '_3순위여유업종', '_3순위여유업종_이용금액', '_1순위납부업종', '_1순위납부업종_이용금액', '_2순위납부업종', '_2순위납부업종_이용금액', '_3순위납부업종', '_3순위납부업종_이용금액', '할부건수_3M_R12M', '할부건수_6M_R12M', '할부건수_12M_R12M', '할부건수_14M_R12M', '할부금액_3M_R12M', '할부금액_6M_R12M', '할부금액_12M_R12M', '할부금액_14M_R12M', '할부건수_유이자_3M_R12M', '할부건수_유이자_6M_R12M', '할부건수_유이자_12M_R12M', '할부건수_유이자_14M_R12M', '할부금액_유이자_3M_R12M', '할부금액_유이자_6M_R12M', '할부금액_유이자_12M_R12M', '할부금액_유이자_14M_R12M', '할부건수_무이자_3M_R12M', '할부건수_무이자_6M_R12M', '할부건수_무이자_12M_R12M', '할부건수_무이자_14M_R12M', '할부금액_무이자_3M_R12M', '할부금액_무이자_6M_R12M', '할부금액_무이자_12M_R12M', '할부금액_무이자_14M_R12M', '할부건수_부분_3M_R12M', '할부건수_부분_6M_R12M', '할부건수_부분_12M_R12M', '할부건수_부분_14M_R12M', '할부금액_부분_3M_R12M', '할부금액_부분_6M_R12M', '할부금액_부분_12M_R12M', '할부금액_부분_14M_R12M']

In [23]:
print(column_list)

['기준년월', 'ID', '이용가맹점수', '이용금액_해외', '쇼핑_도소매_이용금액', '쇼핑_백화점_이용금액', '쇼핑_마트_이용금액', '쇼핑_슈퍼마켓_이용금액', '쇼핑_편의점_이용금액', '쇼핑_아울렛_이용금액', '쇼핑_온라인_이용금액', '쇼핑_기타_이용금액', '교통_주유이용금액', '교통_정비이용금액', '교통_통행료이용금액', '교통_버스지하철이용금액', '교통_택시이용금액', '교통_철도버스이용금액', '여유_운동이용금액', '여유_Pet이용금액', '여유_공연이용금액', '여유_공원이용금액', '여유_숙박이용금액', '여유_여행이용금액', '여유_항공이용금액', '여유_기타이용금액', '납부_통신비이용금액', '납부_관리비이용금액', '납부_렌탈료이용금액', '납부_가스전기료이용금액', '납부_보험료이용금액', '납부_유선방송이용금액', '납부_건강연금이용금액', '납부_기타이용금액', '_1순위업종', '_1순위업종_이용금액', '_2순위업종', '_2순위업종_이용금액', '_3순위업종', '_3순위업종_이용금액', '_1순위쇼핑업종', '_1순위쇼핑업종_이용금액', '_2순위쇼핑업종', '_2순위쇼핑업종_이용금액', '_3순위쇼핑업종', '_3순위쇼핑업종_이용금액', '_1순위교통업종', '_1순위교통업종_이용금액', '_2순위교통업종', '_2순위교통업종_이용금액', '_3순위교통업종', '_3순위교통업종_이용금액', '_1순위여유업종', '_1순위여유업종_이용금액', '_2순위여유업종', '_2순위여유업종_이용금액', '_3순위여유업종', '_3순위여유업종_이용금액', '_1순위납부업종', '_1순위납부업종_이용금액', '_2순위납부업종', '_2순위납부업종_이용금액', '_3순위납부업종', '_3순위납부업종_이용금액', '할부건수_3M_R12M', '할부건수_6M_R12M', '할부건수_12M_R12M', '할부건수_14M_R12M', '할부금액_3M_R12M', '할부금액_6M_R12M', '할부금액_12M_R12M', '할부

In [24]:
df1_train = df1_train[column_list]

In [25]:
df2_train = df2_train[column_list]
df3_train = df3_train[column_list]
df4_train = df4_train[column_list]
df5_train = df5_train[column_list]
df6_train = df6_train[column_list]

In [26]:
combined_df = pd.concat([df1_train, df2_train, df3_train, df4_train, df5_train, df6_train],
                         axis=0,        # 행 방향
                         ignore_index=True)  # 인덱스 초기화

In [34]:
segment_df = segment_df['Segment']

In [56]:
# 세그먼트와 데이터를 합쳐준다.

all_df = pd.concat([segment_df,combined_df], axis = 1)
all_df

,Segment,기준년월,ID,이용가맹점수,이용금액_해외,쇼핑_도소매_이용금액,쇼핑_백화점_이용금액,쇼핑_마트_이용금액,쇼핑_슈퍼마켓_이용금액,쇼핑_편의점_이용금액,...,할부금액_무이자_12M_R12M,할부금액_무이자_14M_R12M,할부건수_부분_3M_R12M,할부건수_부분_6M_R12M,할부건수_부분_12M_R12M,할부건수_부분_14M_R12M,할부금액_부분_3M_R12M,할부금액_부분_6M_R12M,할부금액_부분_12M_R12M,할부금액_부분_14M_R12M
0,D,201807,TRAIN_000000,6,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,E,201807,TRAIN_000001,32,0,645,0,0,435,315,...,0,0,0,0,0,0,0,0,0,0
2,C,201807,TRAIN_000002,27,0,1038,751,924,504,239,...,0,0,0,0,0,0,0,0,0,0
3,D,201807,TRAIN_000003,10,0,0,0,801,487,0,...,0,0,0,0,0,0,0,0,0,0
4,E,201807,TRAIN_000004,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,E,201812,TRAIN_399995,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2399996,D,201812,TRAIN_399996,15,0,1496,0,0,714,0,...,0,0,0,0,0,0,0,0,0,0
2399997,C,201812,TRAIN_399997,27,0,0,0,0,400,300,...,0,0,0,0,0,0,0,0,0,0
2399998,E,201812,TRAIN_399998,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
print(all_df.isna().sum().to_string())

기준년월                       0
ID                         0
Segment                    0
이용가맹점수                     0
이용금액_해외                    0
쇼핑_도소매_이용금액                0
쇼핑_백화점_이용금액                0
쇼핑_마트_이용금액                 0
쇼핑_슈퍼마켓_이용금액               0
쇼핑_편의점_이용금액                0
쇼핑_아울렛_이용금액                0
쇼핑_온라인_이용금액                0
쇼핑_기타_이용금액                 0
교통_주유이용금액                  0
교통_정비이용금액                  0
교통_통행료이용금액                 0
교통_버스지하철이용금액               0
교통_택시이용금액                  0
교통_철도버스이용금액                0
여유_운동이용금액                  0
여유_Pet이용금액                 0
여유_공연이용금액                  0
여유_공원이용금액                  0
여유_숙박이용금액                  0
여유_여행이용금액                  0
여유_항공이용금액                  0
여유_기타이용금액                  0
납부_통신비이용금액                 0
납부_관리비이용금액                 0
납부_렌탈료이용금액                 0
납부_가스전기료이용금액               0
납부_보험료이용금액                 0
납부_유선방송이용금액                0
납부_건강연금이용금액                0
납부_기타이용금액     

_1순위업종                539992
_2순위업종                912725
_3순위업종               1107898
_1순위쇼핑업종              922663
_2순위쇼핑업종             1135042
_3순위쇼핑업종             1312267
_1순위교통업종             1164494
_2순위교통업종             1656423
_3순위교통업종             2045455
_1순위여유업종             1987260
_2순위여유업종             2302286
_3순위여유업종             2377725
_1순위납부업종             1216263
_2순위납부업종             2033640
_3순위납부업종             2310187

In [ ]:
for col in column_list:
    try:
        var_value = all_df[col].var()
        print(f"{col}: {var_value:.2f}")
    except TypeError:
        print(f"❌ 분산 계산 불가 (문자열 포함): {col}")


이용가맹점수: 620.33
이용금액_해외: 244947.96
쇼핑_도소매_이용금액: 544484.29
쇼핑_백화점_이용금액: 77740.47
쇼핑_마트_이용금액: 338766.64
쇼핑_슈퍼마켓_이용금액: 228902.32
쇼핑_편의점_이용금액: 114447.16
쇼핑_아울렛_이용금액: 21936.57
쇼핑_온라인_이용금액: 18827604.18
쇼핑_기타_이용금액: 21235.64
교통_주유이용금액: 1163692.50
교통_정비이용금액: 38359.46
교통_통행료이용금액: 2.75
교통_버스지하철이용금액: 128310.55
교통_택시이용금액: 13153.34
교통_철도버스이용금액: 18573.09
여유_운동이용금액: 99932.42
여유_Pet이용금액: 3211.95
여유_공연이용금액: 2226.47
여유_공원이용금액: 103.48
여유_숙박이용금액: 3802.57
여유_여행이용금액: 0.00
여유_항공이용금액: 18086.05
여유_기타이용금액: 1654.66
납부_통신비이용금액: 1110276.92
납부_관리비이용금액: 656954.84
납부_렌탈료이용금액: 0.00
납부_가스전기료이용금액: 7593.75
납부_보험료이용금액: 423363.85
납부_유선방송이용금액: 0.00
납부_건강연금이용금액: 0.00
납부_기타이용금액: 8414.30
❌ 분산 계산 불가 (문자열 포함): _1순위업종
_1순위업종_이용금액: 78376831.02
❌ 분산 계산 불가 (문자열 포함): _2순위업종
_2순위업종_이용금액: 8171160.65
❌ 분산 계산 불가 (문자열 포함): _3순위업종
_3순위업종_이용금액: 2717267.15
❌ 분산 계산 불가 (문자열 포함): _1순위쇼핑업종
_1순위쇼핑업종_이용금액: 18248816.94
❌ 분산 계산 불가 (문자열 포함): _2순위쇼핑업종
_2순위쇼핑업종_이용금액: 441931.88
❌ 분산 계산 불가 (문자열 포함): _3순위쇼핑업종
_3순위쇼핑업종_이용금액: 256998.47
❌ 분산 계산 불가 (문자열 포함): _1

In [ ]:
all_df['할부금액_부분_3M_R12M'].value_counts()

할부금액_부분_3M_R12M
0    2400000
Name: count, dtype: int64

In [ ]:
all_df['여유_여행이용금액'].value_counts()

여유_여행이용금액
0    2400000
Name: count, dtype: int64

- 분산이 0인 것들은 [할부건수_14M_R12M, 할부건수_유이자_14M_R12M, 할부건수_부분_12M_R12M, 할부건수_무이자_14M_R12M] 컬럼을 제외하면
- 모두 0 하나만을 데이터로 가지고 있다.
- [여유_여행이용금액, 납부_렌탈료이용금액, 납부_유선방송이용금액, 납부_건강연금이용금액, 할부건수_부분_3M_R12M, 할부건수_부분_6M_R12M, 할부건수_부분_14M_R12M, 할부금액_부분_3M_R12M]의 경우 0 하나만을 가진 데이터값이다.

- 0 하나만 가진 데이터값은 상관관계 분석이나 추후 학습에 방해되므로 drop

In [28]:
all_df.drop(['여유_여행이용금액', '납부_렌탈료이용금액', '납부_유선방송이용금액',
             '납부_건강연금이용금액', '할부건수_부분_3M_R12M', '할부건수_부분_6M_R12M', '할부건수_부분_14M_R12M'],
            axis=1, inplace=True)


In [29]:

all_df.drop(['할부금액_부분_3M_R12M'],
            axis=1, inplace=True)


### 변수간 관계파악
Anova -> ETA제곱
카이제곱 -> 크레이머스V

In [46]:
print(column_list)

['기준년월', 'ID', '이용가맹점수', '이용금액_해외', '쇼핑_도소매_이용금액', '쇼핑_백화점_이용금액', '쇼핑_마트_이용금액', '쇼핑_슈퍼마켓_이용금액', '쇼핑_편의점_이용금액', '쇼핑_아울렛_이용금액', '쇼핑_온라인_이용금액', '쇼핑_기타_이용금액', '교통_주유이용금액', '교통_정비이용금액', '교통_통행료이용금액', '교통_버스지하철이용금액', '교통_택시이용금액', '교통_철도버스이용금액', '여유_운동이용금액', '여유_Pet이용금액', '여유_공연이용금액', '여유_공원이용금액', '여유_숙박이용금액', '여유_여행이용금액', '여유_항공이용금액', '여유_기타이용금액', '납부_통신비이용금액', '납부_관리비이용금액', '납부_렌탈료이용금액', '납부_가스전기료이용금액', '납부_보험료이용금액', '납부_유선방송이용금액', '납부_건강연금이용금액', '납부_기타이용금액', '_1순위업종', '_1순위업종_이용금액', '_2순위업종', '_2순위업종_이용금액', '_3순위업종', '_3순위업종_이용금액', '_1순위쇼핑업종', '_1순위쇼핑업종_이용금액', '_2순위쇼핑업종', '_2순위쇼핑업종_이용금액', '_3순위쇼핑업종', '_3순위쇼핑업종_이용금액', '_1순위교통업종', '_1순위교통업종_이용금액', '_2순위교통업종', '_2순위교통업종_이용금액', '_3순위교통업종', '_3순위교통업종_이용금액', '_1순위여유업종', '_1순위여유업종_이용금액', '_2순위여유업종', '_2순위여유업종_이용금액', '_3순위여유업종', '_3순위여유업종_이용금액', '_1순위납부업종', '_1순위납부업종_이용금액', '_2순위납부업종', '_2순위납부업종_이용금액', '_3순위납부업종', '_3순위납부업종_이용금액', '할부건수_3M_R12M', '할부건수_6M_R12M', '할부건수_12M_R12M', '할부건수_14M_R12M', '할부금액_3M_R12M', '할부금액_6M_R12M', '할부금액_12M_R12M', '할부

In [47]:
# 크레이머스V와 eta를 구하기 위한 함수
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(k, r) - 1)))

def eta_squared(anova_ss_between, total_ss):
    return anova_ss_between / total_ss if total_ss != 0 else np.nan

In [48]:
# 수치형/범주형 자동 구분
num_cols = all_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = all_df.select_dtypes(exclude=[np.number]).columns.tolist()

In [49]:
print(cat_cols)

['Segment', 'ID', '_1순위업종', '_2순위업종', '_3순위업종', '_1순위쇼핑업종', '_2순위쇼핑업종', '_3순위쇼핑업종', '_1순위교통업종', '_2순위교통업종', '_3순위교통업종', '_1순위여유업종', '_2순위여유업종', '_3순위여유업종', '_1순위납부업종', '_2순위납부업종', '_3순위납부업종']


In [50]:
print(num_cols)

['기준년월', '이용가맹점수', '이용금액_해외', '쇼핑_도소매_이용금액', '쇼핑_백화점_이용금액', '쇼핑_마트_이용금액', '쇼핑_슈퍼마켓_이용금액', '쇼핑_편의점_이용금액', '쇼핑_아울렛_이용금액', '쇼핑_온라인_이용금액', '쇼핑_기타_이용금액', '교통_주유이용금액', '교통_정비이용금액', '교통_통행료이용금액', '교통_버스지하철이용금액', '교통_택시이용금액', '교통_철도버스이용금액', '여유_운동이용금액', '여유_Pet이용금액', '여유_공연이용금액', '여유_공원이용금액', '여유_숙박이용금액', '여유_여행이용금액', '여유_항공이용금액', '여유_기타이용금액', '납부_통신비이용금액', '납부_관리비이용금액', '납부_렌탈료이용금액', '납부_가스전기료이용금액', '납부_보험료이용금액', '납부_유선방송이용금액', '납부_건강연금이용금액', '납부_기타이용금액', '_1순위업종_이용금액', '_2순위업종_이용금액', '_3순위업종_이용금액', '_1순위쇼핑업종_이용금액', '_2순위쇼핑업종_이용금액', '_3순위쇼핑업종_이용금액', '_1순위교통업종_이용금액', '_2순위교통업종_이용금액', '_3순위교통업종_이용금액', '_1순위여유업종_이용금액', '_2순위여유업종_이용금액', '_3순위여유업종_이용금액', '_1순위납부업종_이용금액', '_2순위납부업종_이용금액', '_3순위납부업종_이용금액', '할부건수_3M_R12M', '할부건수_6M_R12M', '할부건수_12M_R12M', '할부건수_14M_R12M', '할부금액_3M_R12M', '할부금액_6M_R12M', '할부금액_12M_R12M', '할부금액_14M_R12M', '할부건수_유이자_3M_R12M', '할부건수_유이자_6M_R12M', '할부건수_유이자_12M_R12M', '할부건수_유이자_14M_R12M', '할부금액_유이자_3M_R12M', '할부금액_유이자_6M_R12M', '할부금액_유이자_12M_R12M', '할부금액_유이자_14M_R12M', '할

In [51]:
del num_cols[0]

In [52]:
print(num_cols)

['이용가맹점수', '이용금액_해외', '쇼핑_도소매_이용금액', '쇼핑_백화점_이용금액', '쇼핑_마트_이용금액', '쇼핑_슈퍼마켓_이용금액', '쇼핑_편의점_이용금액', '쇼핑_아울렛_이용금액', '쇼핑_온라인_이용금액', '쇼핑_기타_이용금액', '교통_주유이용금액', '교통_정비이용금액', '교통_통행료이용금액', '교통_버스지하철이용금액', '교통_택시이용금액', '교통_철도버스이용금액', '여유_운동이용금액', '여유_Pet이용금액', '여유_공연이용금액', '여유_공원이용금액', '여유_숙박이용금액', '여유_여행이용금액', '여유_항공이용금액', '여유_기타이용금액', '납부_통신비이용금액', '납부_관리비이용금액', '납부_렌탈료이용금액', '납부_가스전기료이용금액', '납부_보험료이용금액', '납부_유선방송이용금액', '납부_건강연금이용금액', '납부_기타이용금액', '_1순위업종_이용금액', '_2순위업종_이용금액', '_3순위업종_이용금액', '_1순위쇼핑업종_이용금액', '_2순위쇼핑업종_이용금액', '_3순위쇼핑업종_이용금액', '_1순위교통업종_이용금액', '_2순위교통업종_이용금액', '_3순위교통업종_이용금액', '_1순위여유업종_이용금액', '_2순위여유업종_이용금액', '_3순위여유업종_이용금액', '_1순위납부업종_이용금액', '_2순위납부업종_이용금액', '_3순위납부업종_이용금액', '할부건수_3M_R12M', '할부건수_6M_R12M', '할부건수_12M_R12M', '할부건수_14M_R12M', '할부금액_3M_R12M', '할부금액_6M_R12M', '할부금액_12M_R12M', '할부금액_14M_R12M', '할부건수_유이자_3M_R12M', '할부건수_유이자_6M_R12M', '할부건수_유이자_12M_R12M', '할부건수_유이자_14M_R12M', '할부금액_유이자_3M_R12M', '할부금액_유이자_6M_R12M', '할부금액_유이자_12M_R12M', '할부금액_유이자_14M_R12M', '할부건수_무이자_

In [53]:
# 제외할 컬럼 리스트
remove_cols = ['여유_여행이용금액', '납부_렌탈료이용금액', '납부_유선방송이용금액',
               '납부_건강연금이용금액', '할부건수_부분_3M_R12M', '할부건수_부분_6M_R12M', '할부건수_부분_14M_R12M', '할부금액_부분_3M_R12M']

# 기존 숫자형 컬럼 리스트에서 제거
num_cols = [col for col in num_cols if col not in remove_cols]

In [54]:
print(num_cols)

['이용가맹점수', '이용금액_해외', '쇼핑_도소매_이용금액', '쇼핑_백화점_이용금액', '쇼핑_마트_이용금액', '쇼핑_슈퍼마켓_이용금액', '쇼핑_편의점_이용금액', '쇼핑_아울렛_이용금액', '쇼핑_온라인_이용금액', '쇼핑_기타_이용금액', '교통_주유이용금액', '교통_정비이용금액', '교통_통행료이용금액', '교통_버스지하철이용금액', '교통_택시이용금액', '교통_철도버스이용금액', '여유_운동이용금액', '여유_Pet이용금액', '여유_공연이용금액', '여유_공원이용금액', '여유_숙박이용금액', '여유_항공이용금액', '여유_기타이용금액', '납부_통신비이용금액', '납부_관리비이용금액', '납부_가스전기료이용금액', '납부_보험료이용금액', '납부_기타이용금액', '_1순위업종_이용금액', '_2순위업종_이용금액', '_3순위업종_이용금액', '_1순위쇼핑업종_이용금액', '_2순위쇼핑업종_이용금액', '_3순위쇼핑업종_이용금액', '_1순위교통업종_이용금액', '_2순위교통업종_이용금액', '_3순위교통업종_이용금액', '_1순위여유업종_이용금액', '_2순위여유업종_이용금액', '_3순위여유업종_이용금액', '_1순위납부업종_이용금액', '_2순위납부업종_이용금액', '_3순위납부업종_이용금액', '할부건수_3M_R12M', '할부건수_6M_R12M', '할부건수_12M_R12M', '할부건수_14M_R12M', '할부금액_3M_R12M', '할부금액_6M_R12M', '할부금액_12M_R12M', '할부금액_14M_R12M', '할부건수_유이자_3M_R12M', '할부건수_유이자_6M_R12M', '할부건수_유이자_12M_R12M', '할부건수_유이자_14M_R12M', '할부금액_유이자_3M_R12M', '할부금액_유이자_6M_R12M', '할부금액_유이자_12M_R12M', '할부금액_유이자_14M_R12M', '할부건수_무이자_3M_R12M', '할부건수_무이자_6M_R12M', '할부건수_무이자_12M_R12M', '할부건수_

In [58]:
target_col = ab_df['Segment']
feature_cols = ab_df.drop('Segment', axis=1)

In [61]:
from scipy.stats import f_oneway

def run_anova(df, target_col, feature_cols):
    """
    feature_cols 리스트에 있는 연속형 변수들을 대상으로
    target_col에 따라 그룹화하여 ANOVA F-test 수행
    """
    results = {}
    for col in feature_cols:
        groups = [df[df[target_col] == label][col] for label in df[target_col].unique()]
        f_stat, p_val = f_oneway(*groups)
        results[col] = {'F': round(f_stat, 4), 'p-value': round(p_val, 4)}
    return pd.DataFrame(results).T.sort_values('p-value')


In [67]:
run_anova(ab_df, 'Segment', num_cols)

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)


,F,p-value
납부_보험료이용금액,17.2635,0.0
납부_통신비이용금액,18.1840,0.0
할부건수_무이자_3M_R12M,88.2156,0.0
할부금액_무이자_3M_R12M,53.3384,0.0
할부건수_무이자_6M_R12M,24.8552,0.0
...,...,...
할부건수_유이자_14M_R12M,NaN,NaN
할부건수_무이자_14M_R12M,NaN,NaN
할부건수_부분_12M_R12M,NaN,NaN
할부금액_부분_6M_R12M,NaN,NaN


In [68]:
run_anova(cde_df, 'Segment', num_cols)

,F,p-value
이용가맹점수,327196.0055,0.0000
이용금액_해외,109376.1864,0.0000
쇼핑_도소매_이용금액,299536.1796,0.0000
쇼핑_백화점_이용금액,98816.4075,0.0000
쇼핑_마트_이용금액,221786.7769,0.0000
...,...,...
할부금액_무이자_14M_R12M,124.4768,0.0000
할부건수_부분_12M_R12M,84.4369,0.0000
할부금액_부분_6M_R12M,2.9344,0.0532
할부건수_무이자_14M_R12M,0.3721,0.6893


### 기존의 코드

In [ ]:
anova = []
chi = []

for col in all_df.columns:
    if col == 'Segment':
        continue

    if col in num_cols:
        data = all_df[[col, 'Segment']].dropna()
        groups = [data[data['Segment'] == val][col] for val in data['Segment'].unique()]
        try:
            stat = f_oneway(*groups).statistic
            ss_between = sum([(g.mean() - data[col].mean())**2 * len(g) for g in groups])
            ss_total = sum((data[col] - data[col].mean())**2)
            eta2 = eta_squared(ss_between, ss_total)
            anova.append({'변수': col, '유형': '수치형', '계수종류': 'Eta²', '상관계수': eta2})
        except:
            continue

    elif col in cat_cols:
        contingency = pd.crosstab(all_df[col], all_df['Segment'])
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            try:
                v = cramers_v(contingency)
                chi.append({'변수': col, '유형': '범주형', '계수종류': "Cramér's V", '상관계수': v})
            except:
                continue

# 결과 정리
result_df1 = pd.DataFrame(anova)
result_df2 = pd.DataFrame(chi)
result_df1 = result_df1.sort_values(by='상관계수', ascending=False).reset_index(drop=True)
result_df2 = result_df2.sort_values(by='상관계수', ascending=False).reset_index(drop=True)

# 결과 출력
display(result_df1)
display(result_df2)

,변수,유형,계수종류,상관계수
0,_3순위업종_이용금액,수치형,Eta²,2.549372e-01
1,_2순위업종_이용금액,수치형,Eta²,2.429217e-01
2,_2순위쇼핑업종_이용금액,수치형,Eta²,2.387084e-01
3,_1순위업종_이용금액,수치형,Eta²,2.198912e-01
4,_3순위쇼핑업종_이용금액,수치형,Eta²,2.172726e-01
...,...,...,...,...
67,할부건수_14M_R12M,수치형,Eta²,1.375602e-05
68,할부금액_부분_6M_R12M,수치형,Eta²,2.446678e-06
69,할부건수_무이자_14M_R12M,수치형,Eta²,3.108322e-07
70,할부금액_부분_14M_R12M,수치형,Eta²,2.398594e-08


,변수,유형,계수종류,상관계수
0,ID,범주형,Cramér's V,1.000000
1,_2순위여유업종,범주형,Cramér's V,0.131715
2,_1순위업종,범주형,Cramér's V,0.116062
3,_3순위여유업종,범주형,Cramér's V,0.105205
4,_2순위업종,범주형,Cramér's V,0.102366
5,_3순위쇼핑업종,범주형,Cramér's V,0.097754
6,_1순위쇼핑업종,범주형,Cramér's V,0.097296
7,_3순위업종,범주형,Cramér's V,0.091734
8,_1순위여유업종,범주형,Cramér's V,0.085033
9,_1순위교통업종,범주형,Cramér's V,0.084260


In [ ]:
pd.set_option('display.max_rows', 100)
display(result_df1)

,변수,유형,계수종류,상관계수
0,_3순위업종_이용금액,수치형,Eta²,2.549372e-01
1,_2순위업종_이용금액,수치형,Eta²,2.429217e-01
2,_2순위쇼핑업종_이용금액,수치형,Eta²,2.387084e-01
3,_1순위업종_이용금액,수치형,Eta²,2.198912e-01
4,_3순위쇼핑업종_이용금액,수치형,Eta²,2.172726e-01
5,이용가맹점수,수치형,Eta²,2.157008e-01
6,쇼핑_도소매_이용금액,수치형,Eta²,2.048195e-01
7,_1순위교통업종_이용금액,수치형,Eta²,1.715343e-01
8,쇼핑_마트_이용금액,수치형,Eta²,1.575289e-01
9,쇼핑_슈퍼마켓_이용금액,수치형,Eta²,1.560192e-01


In [ ]:
result_df1['상관계수'] = result_df1['상관계수'].round(4)

In [ ]:
display(result_df1)

,변수,유형,계수종류,상관계수
0,_3순위업종_이용금액,수치형,Eta²,0.2549
1,_2순위업종_이용금액,수치형,Eta²,0.2429
2,_2순위쇼핑업종_이용금액,수치형,Eta²,0.2387
3,_1순위업종_이용금액,수치형,Eta²,0.2199
4,_3순위쇼핑업종_이용금액,수치형,Eta²,0.2173
5,이용가맹점수,수치형,Eta²,0.2157
6,쇼핑_도소매_이용금액,수치형,Eta²,0.2048
7,_1순위교통업종_이용금액,수치형,Eta²,0.1715
8,쇼핑_마트_이용금액,수치형,Eta²,0.1575
9,쇼핑_슈퍼마켓_이용금액,수치형,Eta²,0.1560


### corr 상관계수

In [ ]:
# all_df에서 기준년월, ID 컬럼은 제거한다.
all_df = all_df.drop(['기준년월','ID'], axis=1)

In [ ]:
cat_cols

['Segment',
 '_1순위업종',
 '_2순위업종',
 '_3순위업종',
 '_1순위쇼핑업종',
 '_2순위쇼핑업종',
 '_3순위쇼핑업종',
 '_1순위교통업종',
 '_2순위교통업종',
 '_3순위교통업종',
 '_1순위여유업종',
 '_2순위여유업종',
 '_3순위여유업종',
 '_1순위납부업종',
 '_2순위납부업종',
 '_3순위납부업종']

In [ ]:
# A가 가장 높은 등급일 것이라 추측하고 세그먼트만 LabelEncoder가 아닌 map으로 진행

segment_order = {'E': 0, 'D': 1, 'C': 2, 'B': 3, 'A': 4}
all_df['Segment_e'] = all_df['Segment'].map(segment_order)

In [ ]:
# 인코딩 진행
# 인코딩할 컬럼만 선택
encoder = LabelEncoder()
all_df[''] = encoder.fit_transform(all_df[''])
all_df[''] = encoder.fit_transform(all_df[''])
all_df[''] = encoder.fit_transform(all_df[''])
all_df[''] = encoder.fit_transform(all_df[''])
all_df[''] = encoder.fit_transform(all_df[''])

In [ ]:
print(all_df['Segment_e'].unique())

[1 0 2 4 3]


In [ ]:
# 비교할 종속변수
target = 'Segment_e'
# 결과를 담을 리스트
corr_result = []
# 반복
for col in num_cols:
    if col in all_df.columns:
        corr = all_df[col].corr(all_df[target])
        corr_result.append({'변수': col, '상관계수': corr})

# 데이터프레임으로 정리한다
result_df3 = pd.DataFrame(corr_result).sort_values(by='상관계수', key=abs, ascending=False).reset_index(drop=True)

display(result_df3)

,변수,상관계수
0,_3순위업종_이용금액,0.504268
1,_2순위업종_이용금액,0.492209
2,_2순위쇼핑업종_이용금액,0.487364
3,_1순위업종_이용금액,0.468099
4,_3순위쇼핑업종_이용금액,0.464139
5,이용가맹점수,0.456553
6,쇼핑_도소매_이용금액,0.452097
7,_1순위교통업종_이용금액,0.408528
8,쇼핑_마트_이용금액,0.395007
9,쇼핑_슈퍼마켓_이용금액,0.390990


In [ ]:
result_AK = pd.concat(
    [result_df1, result_df2],
    axis=0,
    ignore_index=True
)
display(result_AK)

,변수,유형,계수종류,상관계수
0,_3순위업종_이용금액,수치형,Eta²,0.254900
1,_2순위업종_이용금액,수치형,Eta²,0.242900
2,_2순위쇼핑업종_이용금액,수치형,Eta²,0.238700
3,_1순위업종_이용금액,수치형,Eta²,0.219900
4,_3순위쇼핑업종_이용금액,수치형,Eta²,0.217300
5,이용가맹점수,수치형,Eta²,0.215700
6,쇼핑_도소매_이용금액,수치형,Eta²,0.204800
7,_1순위교통업종_이용금액,수치형,Eta²,0.171500
8,쇼핑_마트_이용금액,수치형,Eta²,0.157500
9,쇼핑_슈퍼마켓_이용금액,수치형,Eta²,0.156000


### A,B 와 CDE나누기

In [33]:
all_df.columns

Index(['기준년월', 'ID', 'Segment', '기준년월', 'ID', '이용가맹점수', '이용금액_해외',
       '쇼핑_도소매_이용금액', '쇼핑_백화점_이용금액', '쇼핑_마트_이용금액', '쇼핑_슈퍼마켓_이용금액',
       '쇼핑_편의점_이용금액', '쇼핑_아울렛_이용금액', '쇼핑_온라인_이용금액', '쇼핑_기타_이용금액', '교통_주유이용금액',
       '교통_정비이용금액', '교통_통행료이용금액', '교통_버스지하철이용금액', '교통_택시이용금액', '교통_철도버스이용금액',
       '여유_운동이용금액', '여유_Pet이용금액', '여유_공연이용금액', '여유_공원이용금액', '여유_숙박이용금액',
       '여유_항공이용금액', '여유_기타이용금액', '납부_통신비이용금액', '납부_관리비이용금액', '납부_가스전기료이용금액',
       '납부_보험료이용금액', '납부_기타이용금액', '_1순위업종', '_1순위업종_이용금액', '_2순위업종',
       '_2순위업종_이용금액', '_3순위업종', '_3순위업종_이용금액', '_1순위쇼핑업종', '_1순위쇼핑업종_이용금액',
       '_2순위쇼핑업종', '_2순위쇼핑업종_이용금액', '_3순위쇼핑업종', '_3순위쇼핑업종_이용금액', '_1순위교통업종',
       '_1순위교통업종_이용금액', '_2순위교통업종', '_2순위교통업종_이용금액', '_3순위교통업종',
       '_3순위교통업종_이용금액', '_1순위여유업종', '_1순위여유업종_이용금액', '_2순위여유업종',
       '_2순위여유업종_이용금액', '_3순위여유업종', '_3순위여유업종_이용금액', '_1순위납부업종',
       '_1순위납부업종_이용금액', '_2순위납부업종', '_2순위납부업종_이용금액', '_3순위납부업종',
       '_3순위납부업종_이용금액', '할부건수_3M_R12M', '할부건수_6M_R12M', '할부건수_12M_R12M',
    

In [57]:
# A, B 클래스만 필터링
ab_df = all_df[all_df['Segment'].isin(['A', 'B'])]

# C, D, E 클래스만 필터링
cde_df = all_df[all_df['Segment'].isin(['C', 'D', 'E'])]

In [37]:
ab_df

,Segment,기준년월,ID,이용가맹점수,이용금액_해외,쇼핑_도소매_이용금액,쇼핑_백화점_이용금액,쇼핑_마트_이용금액,쇼핑_슈퍼마켓_이용금액,쇼핑_편의점_이용금액,...,할부금액_무이자_12M_R12M,할부금액_무이자_14M_R12M,할부건수_부분_3M_R12M,할부건수_부분_6M_R12M,할부건수_부분_12M_R12M,할부건수_부분_14M_R12M,할부금액_부분_3M_R12M,할부금액_부분_6M_R12M,할부금액_부분_12M_R12M,할부금액_부분_14M_R12M
2898,A,201807,TRAIN_002898,43,0,3742,0,0,1083,381,...,0,0,0,0,0,0,0,0,0,0
5253,A,201807,TRAIN_005253,28,0,2216,986,1464,700,210,...,0,0,0,0,0,0,0,0,0,0
8128,A,201807,TRAIN_008128,80,1761,2272,975,0,894,519,...,4374,30882,0,0,0,0,0,0,0,0
10808,A,201807,TRAIN_010808,82,2716,3767,1297,2405,1364,447,...,0,0,0,0,0,0,0,0,0,0
14951,A,201807,TRAIN_014951,67,3490,6441,1619,1903,665,152,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2376373,A,201812,TRAIN_376373,86,1722,4897,1604,0,1055,490,...,0,0,0,0,0,0,0,0,0,0
2378479,A,201812,TRAIN_378479,102,1378,3426,1563,3279,1302,930,...,0,0,0,0,0,0,0,0,0,0
2390620,B,201812,TRAIN_390620,6,0,1425,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2393027,A,201812,TRAIN_393027,69,0,3641,0,2286,684,340,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
cde_df

,ID,Segment,_1순위업종,_2순위업종,_3순위업종,_1순위쇼핑업종,_2순위쇼핑업종,_3순위쇼핑업종,_1순위교통업종,_2순위교통업종,...,할부건수_무이자_12M_R12M,할부건수_무이자_14M_R12M,할부금액_무이자_3M_R12M,할부금액_무이자_6M_R12M,할부금액_무이자_12M_R12M,할부금액_무이자_14M_R12M,할부건수_부분_12M_R12M,할부금액_부분_6M_R12M,할부금액_부분_12M_R12M,할부금액_부분_14M_R12M
0,TRAIN_000000,D,납부,쇼핑,사교활동,쇼핑기타,None,None,택시,버스지하철,...,0,0,1314,4514,0,0,0,0,0,0
1,TRAIN_000001,E,쇼핑,납부,교통,도소매,슈퍼마켓,편의점,주유,철도버스,...,0,0,2076,0,0,0,0,0,0,0
2,TRAIN_000002,C,쇼핑,사교활동,교통,온라인,도소매,마트,주유,None,...,0,0,0,10561,0,0,0,0,0,0
3,TRAIN_000003,D,쇼핑,납부,사교활동,마트,슈퍼마켓,None,택시,None,...,0,0,12098,0,0,0,0,0,0,0
4,TRAIN_000004,E,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,TRAIN_399995,E,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
2399996,TRAIN_399996,D,쇼핑,의료,사교활동,도소매,슈퍼마켓,쇼핑기타,버스지하철,택시,...,0,0,1888,0,0,0,0,0,0,0
2399997,TRAIN_399997,C,쇼핑,사교활동,교통,온라인,슈퍼마켓,쇼핑기타,주유,정비,...,0,0,27617,0,0,0,0,0,0,0
2399998,TRAIN_399998,E,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0


In [41]:
anova = []
chi = []

for col in ab_df.columns:
    if col == 'Segment':
        continue

    if col in num_cols:
        data = ab_df[[col, 'Segment']].dropna()
        groups = [data[data['Segment'] == val][col] for val in data['Segment'].unique()]
        try:
            stat = f_oneway(*groups).statistic
            ss_between = sum([(g.mean() - data[col].mean())**2 * len(g) for g in groups])
            ss_total = sum((data[col] - data[col].mean())**2)
            eta2 = eta_squared(ss_between, ss_total)
            anova.append({'변수': col, '유형': '수치형', '계수종류': 'Eta²', '상관계수': eta2})
        except:
            continue

    elif col in cat_cols:
        contingency = pd.crosstab(ab_df[col], ab_df['Segment'])
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            try:
                v = cramers_v(contingency)
                chi.append({'변수': col, '유형': '범주형', '계수종류': "Cramér's V", '상관계수': v})
            except:
                continue

# 결과 정리
result_df11 = pd.DataFrame(anova)
result_df12 = pd.DataFrame(chi)
result_df11 = result_df11.sort_values(by='상관계수', ascending=False).reset_index(drop=True)
result_df12 = result_df12.sort_values(by='상관계수', ascending=False).reset_index(drop=True)

# 결과 출력
display(result_df11)
display(result_df12)

NameError: name 'num_cols' is not defined

In [ ]:
anova = []
chi = []

for col in cde_df.columns:
    if col == 'Segment':
        continue

    if col in num_cols:
        data = cde_df[[col, 'Segment']].dropna()
        groups = [data[data['Segment'] == val][col] for val in data['Segment'].unique()]
        try:
            stat = f_oneway(*groups).statistic
            ss_between = sum([(g.mean() - data[col].mean())**2 * len(g) for g in groups])
            ss_total = sum((data[col] - data[col].mean())**2)
            eta2 = eta_squared(ss_between, ss_total)
            anova.append({'변수': col, '유형': '수치형', '계수종류': 'Eta²', '상관계수': eta2})
        except:
            continue

    elif col in cat_cols:
        contingency = pd.crosstab(cde_df[col], cde_df['Segment'])
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            try:
                v = cramers_v(contingency)
                chi.append({'변수': col, '유형': '범주형', '계수종류': "Cramér's V", '상관계수': v})
            except:
                continue

# 결과 정리
result_df21 = pd.DataFrame(anova)
result_df22 = pd.DataFrame(chi)
result_df21 = result_df21.sort_values(by='상관계수', ascending=False).reset_index(drop=True)
result_df22 = result_df22.sort_values(by='상관계수', ascending=False).reset_index(drop=True)

# 결과 출력
display(result_df21)
display(result_df22)

,변수,유형,계수종류,상관계수
0,_3순위업종_이용금액,수치형,Eta²,2.509779e-01
1,_2순위업종_이용금액,수치형,Eta²,2.390016e-01
2,_2순위쇼핑업종_이용금액,수치형,Eta²,2.345792e-01
3,_3순위쇼핑업종_이용금액,수치형,Eta²,2.150143e-01
4,_1순위업종_이용금액,수치형,Eta²,2.150013e-01
5,이용가맹점수,수치형,Eta²,2.143248e-01
6,쇼핑_도소매_이용금액,수치형,Eta²,1.998271e-01
7,_1순위교통업종_이용금액,수치형,Eta²,1.705684e-01
8,쇼핑_마트_이용금액,수치형,Eta²,1.560530e-01
9,쇼핑_슈퍼마켓_이용금액,수치형,Eta²,1.546159e-01


,변수,유형,계수종류,상관계수
0,ID,범주형,Cramér's V,1.000000
1,_2순위여유업종,범주형,Cramér's V,0.173881
2,_1순위업종,범주형,Cramér's V,0.163681
3,_2순위업종,범주형,Cramér's V,0.143127
4,_1순위쇼핑업종,범주형,Cramér's V,0.137188
5,_3순위쇼핑업종,범주형,Cramér's V,0.136489
6,_3순위여유업종,범주형,Cramér's V,0.134254
7,_3순위업종,범주형,Cramér's V,0.127779
8,_1순위교통업종,범주형,Cramér's V,0.118610
9,_1순위여유업종,범주형,Cramér's V,0.113846


In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
result_df1['상관계수'] = result_df1['상관계수'].round(4)
result_df11['상관계수'] = result_df11['상관계수'].round(4)
result_df21['상관계수'] = result_df21['상관계수'].round(4)

In [ ]:
result_df1

,변수,유형,계수종류,상관계수
0,_3순위업종_이용금액,수치형,Eta²,0.2549
1,_2순위업종_이용금액,수치형,Eta²,0.2429
2,_2순위쇼핑업종_이용금액,수치형,Eta²,0.2387
3,_1순위업종_이용금액,수치형,Eta²,0.2199
4,_3순위쇼핑업종_이용금액,수치형,Eta²,0.2173
5,이용가맹점수,수치형,Eta²,0.2157
6,쇼핑_도소매_이용금액,수치형,Eta²,0.2048
7,_1순위교통업종_이용금액,수치형,Eta²,0.1715
8,쇼핑_마트_이용금액,수치형,Eta²,0.1575
9,쇼핑_슈퍼마켓_이용금액,수치형,Eta²,0.1560


In [ ]:
result_df1

,변수,유형,계수종류,상관계수
0,_3순위업종_이용금액,수치형,Eta²,0.2549
1,_2순위업종_이용금액,수치형,Eta²,0.2429
2,_2순위쇼핑업종_이용금액,수치형,Eta²,0.2387
3,_1순위업종_이용금액,수치형,Eta²,0.2199
4,_3순위쇼핑업종_이용금액,수치형,Eta²,0.2173
5,이용가맹점수,수치형,Eta²,0.2157
6,쇼핑_도소매_이용금액,수치형,Eta²,0.2048
7,_1순위교통업종_이용금액,수치형,Eta²,0.1715
8,쇼핑_마트_이용금액,수치형,Eta²,0.1575
9,쇼핑_슈퍼마켓_이용금액,수치형,Eta²,0.1560


In [ ]:
result_df21

,변수,유형,계수종류,상관계수
0,_3순위업종_이용금액,수치형,Eta²,0.2510
1,_2순위업종_이용금액,수치형,Eta²,0.2390
2,_2순위쇼핑업종_이용금액,수치형,Eta²,0.2346
3,_3순위쇼핑업종_이용금액,수치형,Eta²,0.2150
4,_1순위업종_이용금액,수치형,Eta²,0.2150
5,이용가맹점수,수치형,Eta²,0.2143
6,쇼핑_도소매_이용금액,수치형,Eta²,0.1998
7,_1순위교통업종_이용금액,수치형,Eta²,0.1706
8,쇼핑_마트_이용금액,수치형,Eta²,0.1561
9,쇼핑_슈퍼마켓_이용금액,수치형,Eta²,0.1546


In [ ]:
ab_df.to_parquet('승인매출ab_df3-2.parquet', index=True)
cde_df.to_parquet('승인매출cde_df3-2.parquet', index=True)
result_df11.to_parquet('승인매출ab_df3-2_설명력.parquet', index=True)
result_df21.to_parquet('승인매출cde_df3-2_설명력.parquet', index=True)

In [38]:
ab_df.to_parquet('승인매출ab_df3-2.parquet', index=False)
cde_df.to_parquet('승인매출cde_df3-2.parquet', index=False)

In [75]:
from scipy.stats import ttest_ind
def cohen_d(x, y):
    """두 그룹 간 효과 크기(Cohen's d) 계산"""
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx - 1)*x.std()**2 + (ny - 1)*y.std()**2) / dof)
    return (x.mean() - y.mean()) / pooled_std

def ttest_ab_multiple_verbose_abs_sorted(df, feature_cols):
    """
    A vs B 그룹 간 연속형 변수들에 대해 t-test, p-value, Cohen's d, 유의성 표시 포함한 결과 반환
    Cohen's d 절댓값 기준 내림차순 정렬
    """
    results = []

    for col in feature_cols:
        group_a = df[df['Segment'] == 'A'][col].dropna()
        group_b = df[df['Segment'] == 'B'][col].dropna()

        t_stat, p_val = ttest_ind(group_a, group_b, equal_var=False)
        d = cohen_d(group_a, group_b)

        # 유의성 표시
        if p_val < 0.001:
            sig = '***'
        elif p_val < 0.01:
            sig = '**'
        elif p_val < 0.05:
            sig = '*'
        else:
            sig = 'ns'

        results.append({
            'feature': col,
            't-statistic': round(t_stat, 4),
            'p-value': round(p_val, 4),
            'Cohen\'s d': round(d, 4),
            'abs(Cohen\'s d)': abs(round(d, 4)),
            'significance': sig
        })

    df_result = pd.DataFrame(results)
    df_result = df_result.sort_values('abs(Cohen\'s d)', ascending=False).reset_index(drop=True)
    df_result = df_result.drop(columns=['abs(Cohen\'s d)'])  # 필요 없으면 제거

    return df_result

In [79]:
ttest_results = ttest_ab_multiple_verbose_abs_sorted(ab_df, num_cols)
ttest_results

/tmp/ipython-input-75-3725380163.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return (x.mean() - y.mean()) / pooled_std


,feature,t-statistic,p-value,Cohen's d,significance
0,할부건수_3M_R12M,-9.9951,0.0,-0.8431,***
1,할부건수_무이자_3M_R12M,-9.9617,0.0,-0.8387,***
2,할부금액_3M_R12M,-8.3124,0.0,-0.6618,***
3,할부금액_무이자_3M_R12M,-8.2449,0.0,-0.6521,***
4,할부건수_무이자_6M_R12M,-4.3309,0.0,-0.4452,***
...,...,...,...,...,...
66,할부건수_유이자_14M_R12M,NaN,NaN,NaN,ns
67,할부건수_무이자_14M_R12M,NaN,NaN,NaN,ns
68,할부건수_부분_12M_R12M,NaN,NaN,NaN,ns
69,할부금액_부분_6M_R12M,NaN,NaN,NaN,ns


In [80]:

pd.set_option('display.float_format', '{:.5f}'.format)

# Cramér's V
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(k, r) - 1)))

# Eta² / t-test
def eta_squared_from_t(t, n1, n2):
    df = n1 + n2 - 2
    return t**2 / (t**2 + df) if df > 0 else np.nan

In [84]:
def effect_sizes(df):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

    num_cols = [col for col in num_cols if col != 'Segment']
    cat_cols = [col for col in cat_cols if col != 'Segment']

    ttest_results = []
    chi_results = []

    for col in df.columns:
        if col == 'Segment':
            continue

        # 수치형 변수
        if col in num_cols:
            data = df[[col, 'Segment']].dropna()
            group1 = data[data['Segment'] == data['Segment'].unique()[0]][col]
            group2 = data[data['Segment'] == data['Segment'].unique()[1]][col]
            try:
                t_stat, _ = ttest_ind(group1, group2, equal_var=False)
                eta2 = eta_squared_from_t(t_stat, len(group1), len(group2))
                ttest_results.append({
                    '변수': col, '유형': '수치형',
                    '계수종류': 'Eta²',
                    '상관계수': eta2
                })
            except:
                continue

        # 범주형 변수
        elif col in cat_cols:
            contingency = pd.crosstab(df[col], df['Segment'])
            if contingency.shape[0] > 1 and contingency.shape[1] > 1:
                try:
                    v = cramers_v(contingency)
                    chi_results.append({
                        '변수': col, '유형': '범주형',
                        '계수종류': "Cramér's V",
                        '상관계수': v
                    })
                except:
                    continue

    result_df1 = pd.DataFrame(ttest_results).sort_values(by='상관계수', ascending=False).reset_index(drop=True)
    result_df2 = pd.DataFrame(chi_results).sort_values(by='상관계수', ascending=False).reset_index(drop=True)

    return result_df1, result_df2


In [83]:
result1, result2 = effect_sizes(ab_df)
display(result1)
display(result2)

,변수,유형,계수종류,상관계수
0,할부건수_3M_R12M,수치형,Eta²,0.08230
1,할부건수_무이자_3M_R12M,수치형,Eta²,0.08179
2,할부금액_3M_R12M,수치형,Eta²,0.05840
3,할부금액_무이자_3M_R12M,수치형,Eta²,0.05751
4,할부금액_유이자_12M_R12M,수치형,Eta²,0.02655
...,...,...,...,...
75,할부건수_부분_12M_R12M,수치형,Eta²,NaN
76,할부건수_부분_14M_R12M,수치형,Eta²,NaN
77,할부금액_부분_3M_R12M,수치형,Eta²,NaN
78,할부금액_부분_6M_R12M,수치형,Eta²,NaN


,변수,유형,계수종류,상관계수
0,ID,범주형,Cramér's V,1.00000
1,_2순위납부업종,범주형,Cramér's V,0.29501
2,_3순위업종,범주형,Cramér's V,0.20882
3,_2순위업종,범주형,Cramér's V,0.15342
4,_1순위업종,범주형,Cramér's V,0.14461
5,_2순위쇼핑업종,범주형,Cramér's V,0.13980
6,_2순위교통업종,범주형,Cramér's V,0.13249
7,_3순위쇼핑업종,범주형,Cramér's V,0.12062
8,_1순위여유업종,범주형,Cramér's V,0.11986
9,_3순위여유업종,범주형,Cramér's V,0.11840


In [85]:
#result1과 result2, 그리고 ttest_results를 각각 parquet으로 저장한다.
result1.to_parquet('/content/drive/MyDrive/승인매출ab_df3-2_설명력.parquet', index=False)
ttest_results.to_parquet('/content/drive/MyDrive/ttest_results3-2.parquet', index=False)